standariser les types et les dates

In [47]:
import pandas as pd
import numpy as np
from datetime import datetime, timezone


def inspect_dataframe(df):
    print("=" * 60)
    print(f"DATA SUMMARY")
    print("=" * 60)
    print(f"Shape : {df.shape[0]} rows  x {df.shape[1]} columns")
    print(f"{df.duplicated().sum()} rows")
    print("="*60)
    print(df.describe())
    print("="*60)
    print(df.dtypes)
    print("="*60)
    print(df.isna().sum())



def to_string(df, *columns):
    return df.assign(** {col: df[col].astype('string').str.strip() for col in columns})


def to_numeric(df, *columns ):
    if not columns:
         raise ValueError("column is empty")
    def convert(serie, errors = "coerce"):
            return (
                pd.to_numeric(
                    serie.astype(str)
                        .str.replace(',', '')
                        .str.strip(),
                    errors)
            )

    return df.assign(** {col: convert(df[col]) for col in columns})


def to_datetime(df, *columns):
    return df.assign(**{col: pd.to_datetime(df[col]) for col in columns})
    



    

In [48]:
#test
df = pd.read_csv("../data/bronze/bronze_meteo_data.csv")
inspect_dataframe(df)

DATA SUMMARY
Shape : 4424 rows  x 12 columns
0 rows
          latitude    longitude  temperature_max  temperature_min  \
count  4424.000000  4424.000000      4424.000000      4424.000000   
mean     32.866035    -6.523086        31.161551        19.632301   
std       1.920049     2.146397         4.184110         2.985023   
min      21.615700   -16.468700        16.400000         8.800000   
25%      31.759175    -7.916975        28.000000        17.700000   
50%      32.947000    -6.336650        31.000000        19.600000   
75%      34.385425    -5.110125        34.300000        21.600000   
max      35.844800    -1.226900        42.900000        29.400000   

       precipitation_sum  precipitation_probability_max  wind_speed_10m_max  \
count         4424.00000                    4424.000000         4424.000000   
mean             0.10934                       5.954792           18.944394   
std              0.57044                      10.691943            5.721212   
min       

In [32]:
df = to_numeric(df, 'weather_code', 'wind_gusts_10m_max', 'latitude', 'precipitation_probability_max')
df['forecast_date'] = to_string(df,'forecast_date')
print(df.head(3))
print(df['forecast_date'].dtype)

         city  country  latitude  longitude forecast_date  temperature_max  \
0  Casablanca  Morocco   33.5992      -7.62    2026-09-15             28.8   
1  Casablanca  Morocco   33.5992      -7.62    2026-09-16             25.9   
2  Casablanca  Morocco   33.5992      -7.62    2026-09-17             25.7   

   temperature_min  precipitation_sum  precipitation_probability_max  \
0             21.6                0.0                              0   
1             21.0                0.0                              0   
2             19.9                0.0                              0   

   wind_speed_10m_max  wind_gusts_10m_max  weather_code  
0                10.7                29.9            45  
1                14.1                36.0            45  
2                12.0                33.8             3  
object


In [64]:
df = to_datetime(df, 'forecast_date')
# print(df.head()


print("Number of rows:", len(df))

print("\nNon-null values:")
print(df.count())

print("\nDuplicated rows:", df.duplicated().sum())

print("\nMissing values:")
print(df.isna().sum())

print("\nMissing values (%):")
print((df.isna().mean() * 100).round(2))


df["temp_category"] = pd.cut(
    df["temperature"],
    bins=[-float("inf"), 0, 10, 20, 34, float("inf")],
    labels=["Very Cold", "Cold", "Cool", "Comfortable", "Hot"]
)



df["precipitation_category"] = pd.cut(
    df["precipitation"],
    bins=[-float("inf"), 0, 2.5, 10, 50, float("inf")],
    labels=["No Rain", "Light", "Moderate", "Heavy", "Very Heavy"]
)


df["wind_category"] = pd.cut(
    df["wind_speed"],
    bins=[-float("inf"), 5, 20, 40, 60, float("inf")],
    labels=["Calm", "Light", "Moderate", "Strong", "Very Strong"]
)









# As suggested in #262 and #228 it would be nice to have a weathercode description that could be translated into different languages and using a lang parameter people get the description in ther language. It could be added when someone chooses to include the weathercode. Here's a suggestion for what could be used for English:
# Code 	Description
# 0 	Clear
# 1 	Mostly Clear
# 2 	Partly Cloudy
# 3 	Cloudy
# 45 	Fog
# 48 	Freezing Fog
# 51 	Light Drizzle
# 53 	Drizzle
# 55 	Heavy Drizzle
# 56 	Light Freezing Drizzle
# 57 	Freezing Drizzle
# 61 	Light Rain
# 63 	Rain
# 65 	Heavy Rain
# 66 	Light Freezing Rain
# 67 	Freezing Rain
# 71 	Light Snow
# 73 	Snow
# 75 	Heavy Snow
# 77 	Snow Grains
# 80 	Light Rain Shower
# 81 	Rain Shower
# 82 	Heavy Rain Shower
# 85 	Snow Shower
# 86 	Heavy Snow Shower
# 95 	Thunderstorm
# 96 	Hailstorm
# 99 	Heavy Hailstorm
#
# Is weathercode 96 and 99 thunderstorms with hail or just stronger thunderstorms? The docs say it's supposed to be hail but I don't think the code calculation reflects that or maybe it does and I don('t know how to read the code properly.'
# (''


def get_weather_score(risk_score, weather_code):
    ref = {
        0: 100, 1: 100, 2: 90, 3: 80,       # Clear / cloudy
        45: 60, 48: 60,                       # Fog
        51: 75, 53: 70, 55: 65,               # Drizzle
        61: 55, 63: 45, 65: 30,               # Rain
        66: 25, 67: 20,                       # Freezing rain
        71: 35, 73: 25, 75: 15, 77: 20,       # Snow
        80: 45, 81: 35, 82: 25,               # Rain showers
        85: 20, 86: 10,                       # Snow showers
        95: 10, 96: 0, 99: 0                  # Thunderstorm / hail
    }
    code_quality = ref.get(weather_code, 50)
    risk_quality = 100 - risk_score
    return round(0.7 * risk_quality + 0.3 * code_quality, 2)


def fix_min_max(df, columns):
    df = df.copy()
    for min, max in columns:
        problem = df[min]>df[max]
        df.loc[problem,[min,max]] = df.loc[problem,[max,min]].to_numpy()
    return df




Number of rows: 4424

Non-null values:
city                             4424
country                          4424
latitude                         4424
longitude                        4424
forecast_date                    4424
temperature_max                  4424
temperature_min                  4424
precipitation_sum                4424
precipitation_probability_max    4424
wind_speed_10m_max               4424
wind_gusts_10m_max               4424
weather_code                     4424
dtype: int64

Duplicated rows: 0

Missing values:
city                             0
country                          0
latitude                         0
longitude                        0
forecast_date                    0
temperature_max                  0
temperature_min                  0
precipitation_sum                0
precipitation_probability_max    0
wind_speed_10m_max               0
wind_gusts_10m_max               0
weather_code                     0
dtype: int64

Missing values (%):
c